# 🔧 Diario DevOps — AetherNet

**Autor:** Andres Felipe Martinez Henao

**Proyecto:** AetherNet IoT & Autonomous Rover — Diario de infraestructura, CI/CD y broker MQTT (Sprint 1-4). Trabajo en equipo — área DevOps.

**Links clave:**
- `docs/DevOps/devops.md` — mapeo académico T1-T6 del PDF
- `docs/DevOps/roadmap-devops.md` — roadmap por bloques + búsquedas Google
- `docs/DevOps/README.md` — checklist capturas/logs
- `docker-compose.yml` — orquestación multi-servicio
- `.github/workflows/ci.yml` — pipeline 6 jobs (arduino-cli 1.5.1)
- `backend/mosquitto/config/mosquitto.conf` + `backend/mosquitto/config/acl.conf` — broker ACL `aethernet/#`

> Trazabilidad: RNF-1.1 (docker-compose), RNF-1.2 (CI/CD arduino-cli), DEVOPS-01..09, RF-2.1 (MQTT topics).


## 1. Mapa infra — docker-compose

`docker-compose.yml` — 3 servicios orquestados + red + volumen (RNF-1.1, DEVOPS-01).

| Servicio | Imagen / build | Puertos | Descripción |
|---|---|---|---|
| **fastapi** | `build: ./backend Dockerfile` `python:3.12-slim` | `8000:8000` | FastAPI **1.0.0-sprint1** (`backend/app/main.py:1`) — 8 endpoints (`routers/events.py:25`), `GET /health` `{"status":"ok/degraded","database":"ok","version":"1.0.0-sprint1"}` |
| **postgres** | `postgres:16-alpine` | `5432:5432` | BD `aethernet` — `healthcheck pg_isready -U aethernet`, volumen `postgres_data:/var/lib/postgresql/data`, `init.sql` read-only |
| **mosquitto** | `eclipse-mosquitto:2.0` | `1883:1883` `9001:9001` | Broker MQTT 1883 + WebSockets 9001 — `mosquitto.conf` + `acl.conf` read-only, `data`/`log` persistentes |

- **Red:** `aethernet-net` `driver: bridge` — FastAPI resuelve `postgres` y `mosquitto` por nombre (`DATABASE_URL`, `MQTT_BROKER_HOST`).
- **Volumen:** `postgres_data` (persistencia BD) + montajes `acl.conf:ro` `mosquitto.conf:ro` `init.sql:ro`.
- **Depends:** `fastapi depends_on postgres: service_healthy` + `mosquitto: service_started` (`docker-compose.yml:49`).

Ver celda 2 — `docker-compose.yml` + `acl.conf` + `mosquitto.conf` preview.


In [ ]:
import pathlib
txt = pathlib.Path("docker-compose.yml").read_text()
print(txt[:800])
print("--- acl.conf ---")
print(pathlib.Path("backend/mosquitto/config/acl.conf").read_text()[:500])
print("--- mosquitto.conf ---")
print(pathlib.Path("backend/mosquitto/config/mosquitto.conf").read_text()[:500])

## 2. Mosquitto ACL aethernet/#

`backend/mosquitto/config/mosquitto.conf:32` → `acl_file /mosquitto/config/acl.conf` — DEVOPS-02 Sprint 1.

- **Listener:** `1883 protocol mqtt` + `9001 protocol websockets` (`allow_anonymous true` LAN) — ver celda 2.
- **Regla canónica Sprint 1:** `topic readwrite aethernet/#` + `topic readwrite $SYS/#` (`acl.conf:35`).
- **Futuro prod:** `allow_anonymous false` + `mosquitto_passwd` + `user gateway-esp32 / app-android` por topic (comentado `acl.conf:41`).

| Topic canónico | Productor | Consumidor | QoS | Requisito |
|---|---|---|---|---|
| `aethernet/rover/command` | App Android `MqttManager:215 publishRoverCommand` | Gateway ESP32 `CE5 CSN15 Ch76` → Rover nRF24 | 0 | RF-3.1 MOV-05/06 |
| `aethernet/rover/telemetry` | Rover UNO → Gateway | App / Backend `POST /api/rover/telemetry` | 0 | RF-3.3 |
| `aethernet/access/command` | App / Gateway | MEGA UART `ACCESS:` | 0 | HU-01 RF-2.2 |
| `aethernet/access/event` | MEGA `uart_protocol.cpp:40` → Gateway | Backend `POST /api/access-events` + App | 0 | HU-01 |
| `aethernet/seguridad/intrusion` | MEGA `laser.cpp:32` `SECURITY:` → Gateway | Node-RED Telegram `intrusion_alert.json` | 0 | HU-02 RF-4.1 |
| `aethernet/system/status` | Gateway `heartbeat` | App / Backend | 0 | DEVOPS-05 |
| `aethernet/mega/status` | MEGA `STATUS:` every 5s | Gateway → `aethernet/mega/status` | 0 | DEVOPS-05 |
| `aethernet/sensor/#` | Rover HC-SR04 EMA α=0.2 | Backend `sensor-events` | 0 | HU-03 |

> Topics definidos en `gateway.ino:68` y `docs/architecture.md:60` — documentados aquí, no inventados. Wildcard `#` = sub-árbol (como `/*` en Express).


## 3. CI/CD 6 jobs

Pipeline `.github/workflows/ci.yml` — **6 jobs** sobre `ubuntu-latest` (FOSS, RNF-3.1) — trigger `push branches ['**']` + `pull_request [main, develop]`.

| Job | Qué valida | Trigger / Depende | Tema PDF |
|---|---|---|---|
| `backend-test` | `ruff check app/` + `mypy app/` + `pytest -v` Python 3.12 cache pip | siempre | T5 pruebas automáticas |
| `firmware-compile` | `arduino-cli 1.5.1` matriz 3: `gateway-esp32 esp32:esp32:esp32` + `mega-access arduino:avr:mega` + `rover-uno arduino:avr:uno` + libs `PubSubClient ArduinoJson@6.21.3 RF24 Keypad Servo NewPing` | siempre | T4+T5 IoT |
| `docker-build` | `docker compose build --no-cache` + `up -d` + `pg_isready healthy` + `curl /health {"status":"ok"}` + `POST/GET 4 endpoints` HU-01..HU-03 RF-3.3 + `down -v` | `needs: [backend-test, firmware-compile]` | T4+T5 despliegue |
| `stats-test` | `pytest` en `stats/` EMA α=0.2 `pytest-cov` | siempre | T5 EST |
| `android-build` | `JDK 17 + Android SDK 37` + `./gradlew :app:assembleDebug` — guard `exists build.gradle.kts` (si no, skip) | — | T5 brecha MOV-12 |
| `security-scan` | `Trivy fs SARIF` → `codeql-action/upload-sarif@v3` GitHub Security | siempre | T5 seguridad |

- **Env global:** `ARDUINO_CLI_VERSION: 1.5.1` pinnada reproducibilidad.
- **Branch protection:** CI verde obligatorio para merge a main (RNF-1.2).
- **Artefactos:** `trivy-results.sarif` en Security tab; `docker-compose ps` y logs en `docker-build`.

Ver `docs/DevOps/devops.md §T5` para mapa completo T5 vs pipeline.


In [ ]:
from pathlib import Path
for p in sorted(Path("docs/logs/firmware_sprint3").glob("T1*.log")):
    print(f"--- {p.name} {p.stat().st_size} bytes ---")
    print(p.read_text()[:600])
    print()

## 4. Logs T1 Docker+MQTT

Logs canónicos Sprint 3 **2026-09-11** — 4 terminales paralelas, copia directa `docker compose` + `mosquitto_sub` + `arduino-cli monitor`.

| Archivo | Contenido | Dónde ver |
|---|---|---|
| `docs/logs/firmware_sprint3/live_2026-09-11/T1_docker_live.log` (160 líneas) | `Uvicorn 0.0.0.0:8000` `GET /health 200 OK` + `POST /api/access-events 201` + `mosquitto 2.0.22 Opening 1883/9001` + `gateway-esp32 aethernet-app-*` conexiones `p2,c1,k15` | Celda 5 preview + `T1_docker_2026-09-11.log` resumen 18 líneas `{"status":"ok","version":"1.0.0-sprint1"}` + `aethernet/rover/command {"left_pwm":120}` |
| `docs/logs/firmware_sprint3/live_2026-09-11/T1_mqtt_live.log` (303 líneas) | `mosquitto_sub -h 192.168.1.14 -t aethernet/# -v` — `aethernet/mega/status door_locked/laser_armed free_ram 7241` + `aethernet/access/event 7c78c98f GRANTED / b8789fb DENIED` + `aethernet/seguridad/intrusion laser-01` + `aethernet/rover/command 255/-255` + `aethernet/rover/telemetry ultrasonic_cm 0-126 ir_* rf_rssi -70` + `aethernet/system/status heartbeat wifi_rssi -63..-71` | Celda 5 + `mosquitto_sub` vivo |
| `docs/logs/firmware_sprint3/T1_docker_2026-09-11.log` | Resumen `docker compose up` `✔ postgres Healthy` `✔ mosquitto Started` `✔ fastapi Started` + `curl /health` | Sprint 1 cierre |

**Logs firmware hermanos (no T1):** `T2_mega_*.log` `T3_gateway_*.log` `T4_rover_*.log` `T_telemetry_invertida.log` — ver `docs/Firmware/notebook/Firmware_Notebook.ipynb §6` para T2/T3/T4 detalle.

**Validación T1 integrada:** `docker compose ps` healthy + `curl http://192.168.1.14:8000/health` + `mosquitto_sub aethernet/#` ráfaga MOV-05/06 — `prd.md:49 <50ms MQTT` `stats/ema_filter.py:32 α=0.2`.


## 📸 Fotos/capturas

> Checklist vivo: `docs/DevOps/README.md` — si falta captura deja `![CAPTURA PENDIENTE]`, no borres la línea.

![CAPTURA PENDIENTE](../../docs/DevOps/capturas/docker-ps.png) — docker compose ps healthy
![CAPTURA PENDIENTE](../../docs/DevOps/capturas/ci-green.png) — GitHub Actions 6 jobs verde
![CAPTURA PENDIENTE](../../docs/DevOps/capturas/health-200.png) — curl http://192.168.1.14:8000/health

- **Cómo tomar:** `docker compose ps` (tres `healthy/running`) + screenshot Actions pestaña `CI/CD Pipeline` 6 checks verdes + `curl -s http://192.168.1.14:8000/health | jq` con `{"status":"ok"}`.
- **Guardar en:** `docs/DevOps/capturas/` — nombres exactos `docker-ps.png` `ci-green.png` `health-200.png` (+ opcional `mosquitto-acl.png` `mosquitto_sub aethernet/#`).
- **Validador:** celda 8 lista `*.png` + tamaño; si vacío imprime `CAPTURAS PENDIENTES — ver docs/DevOps/README.md`.


In [ ]:
from pathlib import Path
for p in sorted(Path("docs/DevOps/capturas").glob("*.png")):
    print(p.name, p.stat().st_size)
if not list(Path("docs/DevOps/capturas").glob("*.png")):
    print("CAPTURAS PENDIENTES — ver docs/DevOps/README.md")

## Siguientes pasos

| Sprint | Tarea | Estado | Depende de |
|---|---|---|---|
| **4** | Script arranque único `DEVOPS-09` `docker compose up` como despliegue LAN FOSS | 🔜 | T1 validado |
| **4** | Health extendido broker vivo + DB `degraded` (`backend/app/main.py:78`) | 🔜 | `GET /health` actual |
| **Post** | Stack `Prometheus+Grafana` T6 notas (`docs/DevOps/devops.md:84`) | 📋 declarado fuera alcance | `system/status` heartbeat ya existe |
| **Siempre** | Tomar capturas `docker-ps.png` `ci-green.png` `health-200.png` → marcar `[x]` en `docs/DevOps/README.md` | 📸 pendiente | este notebook celda 7/8 |

**Refs:**
- `docs/DevOps/devops.md` — T1~T6 mapa académico
- `docs/DevOps/roadmap-devops.md` — Bloques 1-7 + búsquedas Google + datasheets (nuevo)
- `docs/DevOps/README.md` — checklist capturas/logs canónicos
- `docs/sprints.md:61` + `docs/backlog.md §Área 2` + `.github/workflows/ci.yml:7` `ARDUINO_CLI_VERSION 1.5.1`
- KPI PRD `prd.md:49 <50ms MQTT / <10ms RF` + `RNF-1.1/1.2` + `DEVOPS-02 ACL` + `RF-2.1 topics`

> Ejecuta celdas 2, 5, 8 para verificar infra / logs T1 / capturas antes de sprint review.
